In [ ]:
import os
import shutil
import torch
from PIL import Image
from torchvision import transforms
from transformers import CLIPProcessor, CLIPModel
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Load CLIP model and processor
model_name = "openai/clip-vit-large-patch14"
model = CLIPModel.from_pretrained(model_name).to(device)
processor = CLIPProcessor.from_pretrained(model_name)

In [ ]:
# import os
# import shutil
# from datasets import load_dataset
# from sklearn.model_selection import train_test_split
# import pandas as pd
# from PIL import Image
# import numpy as np
# from tqdm import tqdm
# import time

# # ============================================
# # 1. LOAD DATASETS FROM ALL THREE SOURCES
# # ============================================

# # Source 1: Hugging Face Dataset
# print("="*60)
# print("LOADING HUGGING FACE DATASET")
# print("="*60)
# print("Loading dataset from Hugging Face...")
# hf_dataset = load_dataset("saakshigupta/deepfake-detection-dataset-v3")
# print("✓ Hugging Face dataset loaded successfully!")

# print("\nExtracting splits...")
# hf_train = hf_dataset["train"]
# hf_test = hf_dataset["test"]
# print(f"✓ Train split: {len(hf_train)} samples")
# print(f"✓ Test split: {len(hf_test)} samples")

# # Source 2: SP-CUP-2025 Dataset (from mdnakib)
# sp_cup_real_dir = "/kaggle/input/datasets/mdnakib/sp-cup-2025/train_real/real"
# sp_cup_fake_dir = "/kaggle/input/datasets/mdnakib/sp-cup-2025/train_fake/fake"

# # Source 3: Deepfake and Real Images Dataset (from manjilkarki)
# df_real_dir = "/kaggle/input/datasets/manjilkarki/deepfake-and-real-images/Dataset/Train/Real"
# df_fake_dir = "/kaggle/input/datasets/manjilkarki/deepfake-and-real-images/Dataset/Train/Fake"

# # ============================================
# # 2. FUNCTION TO LOAD IMAGES FROM DIRECTORIES
# # ============================================

# def load_images_from_directory(directory, label, dataset_name=""):
#     """Load images from a directory and return as list with labels"""
#     images = []
#     labels = []
    
#     if os.path.exists(directory):
#         # Get all image files
#         files = [f for f in os.listdir(directory) 
#                 if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]
        
#         print(f"\n📂 Loading from: {directory}")
#         print(f"   Found {len(files)} images")
        
#         # Use tqdm for progress bar
#         for filename in tqdm(files, desc=f"   Loading {dataset_name}", unit="images"):
#             img_path = os.path.join(directory, filename)
#             try:
#                 img = Image.open(img_path).convert('RGB')
#                 images.append(img)
#                 labels.append(label)
#             except Exception as e:
#                 print(f"   ⚠️ Error loading {filename}: {e}")
#     else:
#         print(f"   ❌ Directory not found: {directory}")
    
#     return images, labels

# # ============================================
# # 3. LOAD IMAGES FROM DIRECTORY DATASETS
# # ============================================

# print("\n" + "="*60)
# print("LOADING SP-CUP-2025 DATASET")
# print("="*60)

# sp_cup_images = []
# sp_cup_labels = []

# # Load real images
# real_images, real_labels = load_images_from_directory(sp_cup_real_dir, 0, "SP-CUP Real")
# sp_cup_images.extend(real_images)
# sp_cup_labels.extend(real_labels)

# # Load fake images
# fake_images, fake_labels = load_images_from_directory(sp_cup_fake_dir, 1, "SP-CUP Fake")
# sp_cup_images.extend(fake_images)
# sp_cup_labels.extend(fake_labels)

# print(f"\n✅ SP-CUP-2025: {len(sp_cup_images)} total images loaded")
# print(f"   Real: {real_images_count if 'real_images' in locals() else len(real_images)}")
# print(f"   Fake: {fake_images_count if 'fake_images' in locals() else len(fake_images)}")

# print("\n" + "="*60)
# print("LOADING DEEPFAKE AND REAL IMAGES DATASET")
# print("="*60)

# df_images = []
# df_labels = []

# # Load real images
# real_images, real_labels = load_images_from_directory(df_real_dir, 0, "Deepfake-Real Real")
# df_images.extend(real_images)
# df_labels.extend(real_labels)

# # Load fake images
# fake_images, fake_labels = load_images_from_directory(df_fake_dir, 1, "Deepfake-Real Fake")
# df_images.extend(fake_images)
# df_labels.extend(fake_labels)

# print(f"\n✅ Deepfake and Real: {len(df_images)} total images loaded")

# # ============================================
# # 4. PROCESS HUGGING FACE DATASET
# # ============================================

# print("\n" + "="*60)
# print("PROCESSING HUGGING FACE DATASET")
# print("="*60)

# print("Extracting images and labels from Hugging Face dataset...")

# # Use tqdm for progress
# hf_train_images = []
# hf_train_labels = []
# for sample in tqdm(hf_train, desc="Processing train", unit="samples"):
#     hf_train_images.append(sample["image"])
#     hf_train_labels.append(sample["label"])

# hf_test_images = []
# hf_test_labels = []
# for sample in tqdm(hf_test, desc="Processing test", unit="samples"):
#     hf_test_images.append(sample["image"])
#     hf_test_labels.append(sample["label"])

# print(f"\n✅ Hugging Face Train: {len(hf_train_images)} images")
# print(f"✅ Hugging Face Test: {len(hf_test_images)} images")

# # ============================================
# # 5. SPLIT HF TEST INTO VALIDATION AND TEST
# # ============================================

# print("\n" + "="*60)
# print("SPLITTING HF TEST INTO VALIDATION AND TEST")
# print("="*60)

# # Split the test set into validation and test (50-50 or adjust as needed)
# test_size = 0.5  # 50% for validation, 50% for test
# print(f"Splitting with test_size={test_size}...")
# hf_val_images, hf_test_images_split, hf_val_labels, hf_test_labels_split = train_test_split(
#     hf_test_images, hf_test_labels, test_size=test_size, random_state=42, stratify=hf_test_labels
# )

# print(f"\n✅ Hugging Face Validation: {len(hf_val_images)} images")
# print(f"✅ Hugging Face Test (after split): {len(hf_test_images_split)} images")

# # ============================================
# # 6. COMBINE ALL DATASETS
# # ============================================

# print("\n" + "="*60)
# print("COMBINING ALL DATASETS")
# print("="*60)

# print("Combining training data...")
# # Combine training data (all datasets except HF test)
# train_images = sp_cup_images + df_images + hf_train_images
# train_labels = sp_cup_labels + df_labels + hf_train_labels

# # Validation data (only HF validation split)
# val_images = hf_val_images
# val_labels = hf_val_labels

# # Test data (only HF test split)
# test_images = hf_test_images_split
# test_labels = hf_test_labels_split

# print(f"\n📊 FINAL DATASET DISTRIBUTION:")
# print(f"   {'Training:':<15} {len(train_images):>6} images")
# print(f"   {'Validation:':<15} {len(val_images):>6} images")
# print(f"   {'Test:':<15} {len(test_images):>6} images")
# print(f"   {'Total:':<15} {len(train_images) + len(val_images) + len(test_images):>6} images")

# # Count labels in training set
# train_real = sum(1 for label in train_labels if label == 0)
# train_fake = sum(1 for label in train_labels if label == 1)
# print(f"\n📊 Training set breakdown:")
# print(f"   {'Real:':<15} {train_real:>6} images")
# print(f"   {'Fake:':<15} {train_fake:>6} images")

# # ============================================
# # 7. SAVE COMBINED DATASET (OPTIONAL)
# # ============================================

# print("\n" + "="*60)
# print("SAVING COMBINED DATASET")
# print("="*60)

# # Create directories for combined dataset
# combined_dir = "/kaggle/working/combined_dataset"
# train_dir = os.path.join(combined_dir, "train")
# val_dir = os.path.join(combined_dir, "val")
# test_dir = os.path.join(combined_dir, "test")

# # Create directories
# for dir_path in [train_dir, val_dir, test_dir]:
#     os.makedirs(dir_path, exist_ok=True)
#     os.makedirs(os.path.join(dir_path, "real"), exist_ok=True)
#     os.makedirs(os.path.join(dir_path, "fake"), exist_ok=True)

# def save_images(images, labels, base_dir, split_name):
#     """Save images to directories based on their labels"""
#     real_count = 0
#     fake_count = 0
    
#     print(f"\n💾 Saving {split_name} set ({len(images)} images)...")
    
#     for idx, (img, label) in enumerate(tqdm(zip(images, labels), desc=f"   Saving {split_name}", unit="images")):
#         label_dir = "real" if label == 0 else "fake"
#         if label == 0:
#             real_count += 1
#         else:
#             fake_count += 1
#         save_path = os.path.join(base_dir, label_dir, f"img_{idx}.jpg")
#         img.save(save_path)
    
#     print(f"   ✅ {split_name} saved: {real_count} real, {fake_count} fake")

# print("\nSaving datasets...")
# save_images(train_images, train_labels, train_dir, "Train")
# save_images(val_images, val_labels, val_dir, "Validation")
# save_images(test_images, test_labels, test_dir, "Test")

# print("\n✅ Combined dataset saved successfully!")
# print(f"   Location: {combined_dir}")

# # ============================================
# # 8. CREATE DATAFRAME WITH PATHS (FOR EASY ACCESS)
# # ============================================

# print("\n" + "="*60)
# print("CREATING DATAFRAME")
# print("="*60)

# def create_dataframe(images, labels, split_name):
#     """Create a pandas DataFrame with image paths and labels"""
#     data = []
#     for img, label in zip(images, labels):
#         data.append({
#             'image': img,
#             'label': label,
#             'split': split_name
#         })
#     return pd.DataFrame(data)

# print("Creating DataFrames...")
# train_df = create_dataframe(train_images, train_labels, 'train')
# val_df = create_dataframe(val_images, val_labels, 'validation')
# test_df = create_dataframe(test_images, test_labels, 'test')

# # Concatenate all DataFrames
# full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

# print(f"\n📊 Full dataset shape: {full_df.shape}")
# print(f"\n📊 Label distribution:")
# label_counts = full_df['label'].value_counts()
# for label, count in label_counts.items():
#     label_name = "Real" if label == 0 else "Fake"
#     percentage = (count / len(full_df)) * 100
#     print(f"   {label_name}: {count:>6} ({percentage:.1f}%)")

# # ============================================
# # 9. UPDATED DIRECTORY DEFINITIONS
# # ============================================

# print("\n" + "="*60)
# print("DATASET READY FOR USE!")
# print("="*60)

# # If you want to work with the saved combined dataset, use these:
# real_dir = "/kaggle/working/combined_dataset/train/real"
# fake_dir = "/kaggle/working/combined_dataset/train/fake"

# print(f"\n📂 Directories for training:")
# print(f"   Real images: {real_dir}")
# print(f"   Fake images: {fake_dir}")

# print(f"\n📂 Validation directory:")
# print(f"   {combined_dir}/val/")
# print(f"\n📂 Test directory:")
# print(f"   {combined_dir}/test/")

# # Display summary
# print("\n" + "="*60)
# print("SUMMARY")
# print("="*60)
# print(f"Total Training Images:  {len(train_images):>6}")
# print(f"Total Validation Images:{len(val_images):>6}")
# print(f"Total Test Images:      {len(test_images):>6}")
# print(f"{'='*60}")
# print("✅ All datasets loaded and combined successfully!")

In [ ]:
!pip install -q datasets

In [ ]:
import os
import shutil
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import pandas as pd
from PIL import Image
import numpy as np
from tqdm import tqdm
import gc

print("="*60)
print("LOADING HUGGING FACE DATASET ONLY")
print("="*60)

# ============================================
# 1. LOAD HUGGING FACE DATASET
# ============================================
print("\n1. Loading Hugging Face dataset...")
try:
    hf_dataset = load_dataset("saakshigupta/deepfake-detection-dataset-v3")
    print("✅ Hugging Face dataset loaded successfully!")
    
    hf_train = hf_dataset["train"]
    hf_test = hf_dataset["test"]
    print(f"   Train split: {len(hf_train)} samples")
    print(f"   Test split: {len(hf_test)} samples")
except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    raise

# ============================================
# 2. SPLIT HF TEST INTO VALIDATION AND TEST
# ============================================
print("\n2. Splitting HF test into validation and test...")

# Extract images and labels from HF test
hf_test_images = []
hf_test_labels = []
for sample in tqdm(hf_test, desc="Processing test", unit="samples"):
    hf_test_images.append(sample["image"])
    hf_test_labels.append(sample["label"])

# Split 50-50 for validation and test
test_size = 0.5  # 50% for validation, 50% for test
hf_val_images, hf_test_images_split, hf_val_labels, hf_test_labels_split = train_test_split(
    hf_test_images, hf_test_labels, test_size=test_size, random_state=42, stratify=hf_test_labels
)

print(f"\n✅ HF Validation: {len(hf_val_images)} images")
print(f"✅ HF Test: {len(hf_test_images_split)} images")

# Free memory
del hf_test_images, hf_test_labels
gc.collect()

# ============================================
# 3. PROCESS HF TRAIN DATA
# ============================================
print("\n3. Processing HF training data...")
hf_train_images = []
hf_train_labels = []

for sample in tqdm(hf_train, desc="Processing train", unit="samples"):
    hf_train_images.append(sample["image"])
    hf_train_labels.append(sample["label"])

print(f"✅ HF Train loaded: {len(hf_train_images)} images")

# ============================================
# 4. CREATE DIRECTORY STRUCTURE
# ============================================
print("\n4. Creating directory structure...")

# Create directories for training (real and fake)
real_dir = "/kaggle/working/balanced_dataset/real"
fake_dir = "/kaggle/working/balanced_dataset/fake"
val_real_dir = "/kaggle/working/balanced_dataset/val/real"
val_fake_dir = "/kaggle/working/balanced_dataset/val/fake"
test_real_dir = "/kaggle/working/balanced_dataset/test/real"
test_fake_dir = "/kaggle/working/balanced_dataset/test/fake"

# Create all directories
for dir_path in [real_dir, fake_dir, val_real_dir, val_fake_dir, test_real_dir, test_fake_dir]:
    os.makedirs(dir_path, exist_ok=True)

print(f"✅ Directories created:")
print(f"   Train Real: {real_dir}")
print(f"   Train Fake: {fake_dir}")
print(f"   Val Real: {val_real_dir}")
print(f"   Val Fake: {val_fake_dir}")
print(f"   Test Real: {test_real_dir}")
print(f"   Test Fake: {test_fake_dir}")

# ============================================
# 5. SAVE IMAGES TO DIRECTORIES
# ============================================
print("\n5. Saving images to directories...")

def save_images_to_dir(images, labels, output_dir, prefix=""):
    """Save images to a directory with progress tracking"""
    saved_count = 0
    
    for idx, (img, label) in enumerate(tqdm(zip(images, labels), desc=f"   Saving {prefix}", unit="images")):
        try:
            # Determine which directory to save to
            if label == 0:  # Real
                save_path = os.path.join(output_dir, "real", f"{prefix}img_{idx:06d}.jpg")
            else:  # Fake
                save_path = os.path.join(output_dir, "fake", f"{prefix}img_{idx:06d}.jpg")
            
            img.save(save_path)
            saved_count += 1
        except Exception as e:
            print(f"   ⚠️ Error saving image {idx}: {e}")
    
    return saved_count

# Save training data to balanced_dataset
print("\n💾 Saving training data...")
train_real_count = 0
train_fake_count = 0

for idx, (img, label) in enumerate(tqdm(zip(hf_train_images, hf_train_labels), desc="Saving train", unit="images")):
    try:
        if label == 0:  # Real
            save_path = os.path.join(real_dir, f"real_{idx:06d}.jpg")
            train_real_count += 1
        else:  # Fake
            save_path = os.path.join(fake_dir, f"fake_{idx:06d}.jpg")
            train_fake_count += 1
        img.save(save_path)
    except Exception as e:
        print(f"   ⚠️ Error saving image {idx}: {e}")

print(f"✅ Training data saved:")
print(f"   Real: {train_real_count} images in {real_dir}")
print(f"   Fake: {train_fake_count} images in {fake_dir}")

# Save validation data
print("\n💾 Saving validation data...")
val_real_count = 0
val_fake_count = 0

for idx, (img, label) in enumerate(tqdm(zip(hf_val_images, hf_val_labels), desc="Saving val", unit="images")):
    try:
        if label == 0:  # Real
            save_path = os.path.join(val_real_dir, f"val_real_{idx:06d}.jpg")
            val_real_count += 1
        else:  # Fake
            save_path = os.path.join(val_fake_dir, f"val_fake_{idx:06d}.jpg")
            val_fake_count += 1
        img.save(save_path)
    except Exception as e:
        print(f"   ⚠️ Error saving image {idx}: {e}")

print(f"✅ Validation data saved:")
print(f"   Real: {val_real_count} images in {val_real_dir}")
print(f"   Fake: {val_fake_count} images in {val_fake_dir}")

# Save test data
print("\n💾 Saving test data...")
test_real_count = 0
test_fake_count = 0

for idx, (img, label) in enumerate(tqdm(zip(hf_test_images_split, hf_test_labels_split), desc="Saving test", unit="images")):
    try:
        if label == 0:  # Real
            save_path = os.path.join(test_real_dir, f"test_real_{idx:06d}.jpg")
            test_real_count += 1
        else:  # Fake
            save_path = os.path.join(test_fake_dir, f"test_fake_{idx:06d}.jpg")
            test_fake_count += 1
        img.save(save_path)
    except Exception as e:
        print(f"   ⚠️ Error saving image {idx}: {e}")

print(f"✅ Test data saved:")
print(f"   Real: {test_real_count} images in {test_real_dir}")
print(f"   Fake: {test_fake_count} images in {test_fake_dir}")

# ============================================
# 6. CREATE DATAFRAMES
# ============================================
print("\n6. Creating DataFrames...")

def create_dataframe(images, labels, split_name):
    """Create a pandas DataFrame with image paths and labels"""
    data = []
    for img, label in zip(images, labels):
        data.append({
            'image': img,
            'label': label,
            'split': split_name
        })
    return pd.DataFrame(data)

# Create DataFrames
train_df = create_dataframe(hf_train_images, hf_train_labels, 'train')
val_df = create_dataframe(hf_val_images, hf_val_labels, 'validation')
test_df = create_dataframe(hf_test_images_split, hf_test_labels_split, 'test')

# Concatenate all DataFrames
full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

# Save DataFrames
df_dir = "/kaggle/working/dataframes"
os.makedirs(df_dir, exist_ok=True)

train_df.to_pickle(os.path.join(df_dir, "train_df.pkl"))
val_df.to_pickle(os.path.join(df_dir, "val_df.pkl"))
test_df.to_pickle(os.path.join(df_dir, "test_df.pkl"))
full_df.to_pickle(os.path.join(df_dir, "full_df.pkl"))

print(f"✅ DataFrames saved to: {df_dir}")

# ============================================
# 7. VERIFY DATASET
# ============================================
print("\n7. Verifying dataset...")

def count_images_in_dir(directory):
    """Count images in a directory"""
    if os.path.exists(directory):
        files = [f for f in os.listdir(directory) 
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        return len(files)
    return 0

print("\n📊 FINAL DATASET DISTRIBUTION:")
print(f"\nTraining Data:")
print(f"   Real: {count_images_in_dir(real_dir)} images")
print(f"   Fake: {count_images_in_dir(fake_dir)} images")
print(f"   Total: {count_images_in_dir(real_dir) + count_images_in_dir(fake_dir)} images")

print(f"\nValidation Data:")
print(f"   Real: {count_images_in_dir(val_real_dir)} images")
print(f"   Fake: {count_images_in_dir(val_fake_dir)} images")
print(f"   Total: {count_images_in_dir(val_real_dir) + count_images_in_dir(val_fake_dir)} images")

print(f"\nTest Data:")
print(f"   Real: {count_images_in_dir(test_real_dir)} images")
print(f"   Fake: {count_images_in_dir(test_fake_dir)} images")
print(f"   Total: {count_images_in_dir(test_real_dir) + count_images_in_dir(test_fake_dir)} images")

print(f"\n📊 Full dataset shape: {full_df.shape}")
print(f"\n📊 Label distribution:")
label_counts = full_df['label'].value_counts()
for label, count in label_counts.items():
    label_name = "Real" if label == 0 else "Fake"
    percentage = (count / len(full_df)) * 100
    print(f"   {label_name}: {count:>6} ({percentage:.1f}%)")

# ============================================
# 8. FINAL DIRECTORIES FOR YOUR CODE
# ============================================
print("\n" + "="*60)
print("DATASET READY FOR USE!")
print("="*60)

# These are the directories your code expects
print(f"\n📂 Training directories:")
print(f"   real_dir = '{real_dir}'")
print(f"   fake_dir = '{fake_dir}'")

print(f"\n📂 Validation directories:")
print(f"   val_real_dir = '{val_real_dir}'")
print(f"   val_fake_dir = '{val_fake_dir}'")

print(f"\n📂 Test directories:")
print(f"   test_real_dir = '{test_real_dir}'")
print(f"   test_fake_dir = '{test_fake_dir}'")

print("\n" + "="*60)
print("✅ Hugging Face dataset loaded and organized successfully!")
print("="*60)

# ============================================
# 9. CLEANUP (Optional)
# ============================================
print("\n🧹 Cleaning up memory...")
del hf_train_images, hf_train_labels, hf_val_images, hf_val_labels
del hf_test_images_split, hf_test_labels_split
gc.collect()
print("✅ Memory cleaned!")

In [ ]:
# # Define directories
# real_dir = "/kaggle/input/datasets/manjilkarki/deepfake-and-real-images/Dataset/Train/Real"
# fake_dir = "/kaggle/input/datasets/manjilkarki/deepfake-and-real-images/Dataset/Train/Fake"
rag_real_dir = "/kaggle/working/balanced_dataset/real"
rag_fake_dir = "/kaggle/working/balanced_dataset/fake"

In [ ]:
# Create new directories if they don't exist
os.makedirs(rag_real_dir, exist_ok=True)
os.makedirs(rag_fake_dir, exist_ok=True)

In [ ]:
# Define image transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [ ]:
def load_and_transform_image(image_path):
    image = Image.open(image_path)
    image = transform(image)
    return image

def extract_features(directory, limit = None):
    features = []
    filenames = []
    files = os.listdir(directory)[:limit] if limit else os.listdir(directory)
    for filename in tqdm(files, desc=f"Processing {directory}"):
        image_path = os.path.join(directory, filename)
        image = load_and_transform_image(image_path).to(device)
        inputs = processor(images=image, return_tensors="pt").to(device)
        outputs = model.get_image_features(**inputs)
        features.append(outputs.cpu().detach().numpy())
        filenames.append(filename)
    return np.vstack(features), filenames


def save_image_pairs(real_image_paths, fake_image_paths, rag_real_dir, rag_fake_dir):
    for idx, (real_image_path, fake_image_path) in enumerate(zip(real_image_paths, fake_image_paths), start=1):
        real_ext = os.path.splitext(real_image_path)[1]
        fake_ext = os.path.splitext(fake_image_path)[1]

        real_image_name = f"real{idx:05d}{real_ext}"
        fake_image_name = f"fake{idx:05d}{fake_ext}"
        
        shutil.copy(real_image_path, os.path.join(rag_real_dir, real_image_name))
        shutil.copy(fake_image_path, os.path.join(rag_fake_dir, fake_image_name))



def display_image_pairs(real_image_paths, fake_image_paths):
    fig, axs = plt.subplots(10, 2, figsize=(10, 40))
    for i in range(10):
        real_image = Image.open(real_image_paths[i])
        fake_image = Image.open(fake_image_paths[i])
        axs[i, 0].imshow(real_image)
        axs[i, 0].set_title("Real Image")
        axs[i, 0].axis("off")
        axs[i, 1].imshow(fake_image)
        axs[i, 1].set_title("Fake Image")
        axs[i, 1].axis("off")
    plt.show()

In [ ]:
# Extract features for real and fake images
real_features, real_filenames = extract_features(real_dir)
fake_features, fake_filenames = extract_features(fake_dir)

In [ ]:
import numpy as np
import gc
from sklearn.metrics.pairwise import cosine_similarity

def process_in_batches(real_features, real_filenames, fake_features, fake_filenames, batch_size=5000):
    top_pairs = []
    num_real_images = len(real_features)
    used_fake_indices = set()  # To keep track of used fake images

    for start in tqdm(range(0, num_real_images, batch_size), desc="Processing Batches"):
        end = min(start + batch_size, num_real_images)
        batch_real_features = real_features[start:end]
        similarity_matrix = cosine_similarity(batch_real_features, fake_features)
        
        # Find the most similar fake image for each real image in the batch
        for real_idx in range(similarity_matrix.shape[0]):
            available_similarities = [(i, sim) for i, sim in enumerate(similarity_matrix[real_idx]) if i not in used_fake_indices]
            if available_similarities:
                fake_idx, _ = max(available_similarities, key=lambda x: x[1])
                used_fake_indices.add(fake_idx)
                top_pairs.append((os.path.join(real_dir, real_filenames[start + real_idx]), os.path.join(fake_dir, fake_filenames[fake_idx])))
        
        # Delete the similarity matrix of the current batch
        del similarity_matrix
        gc.collect()
    
    return top_pairs
# Process in batches and find the most similar image pairs 
top_pairs = process_in_batches(real_features, real_filenames, fake_features, fake_filenames, batch_size=1000) 
print(len(top_pairs))

In [ ]:
print(len(real_filenames))
print(len(fake_filenames))

In [ ]:
# Save the matched image pairs to the new directories
real_image_paths, fake_image_paths = zip(*top_pairs)
save_image_pairs(real_image_paths, fake_image_paths, rag_real_dir, rag_fake_dir)

In [ ]:
# Display the top 20 pairs
display_image_pairs(real_image_paths[:10], fake_image_paths[:10])

In [ ]:
import os

# Specify the directory
base_dir = "/kaggle/working/balanced_dataset"

# Iterate over each folder in the base directory
for folder_name in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder_name)
    if os.path.isdir(folder_path):
        num_files = len(os.listdir(folder_path))
        print(f"Folder: {folder_name}, Number of files: {num_files}")

In [ ]:
import matplotlib.pyplot as plt
import os
from PIL import Image

def display_first_5_images(fake_dir, real_dir):
    # Get list of image files in the directories
    fake_images = sorted([os.path.join(fake_dir, img) for img in os.listdir(fake_dir)])[:5]
    real_images = sorted([os.path.join(real_dir, img) for img in os.listdir(real_dir)])[:5]
    
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    fig.suptitle('First 5 Images of Fake and Real Directories', fontsize=16)

    # Display fake images
    for i, img_path in enumerate(fake_images):
        img = Image.open(img_path)
        axes[0, i].imshow(img)
        axes[0, i].set_title(f'Fake Image {i+1}')
        axes[0, i].axis('off')
    
    # Display real images
    for i, img_path in enumerate(real_images):
        img = Image.open(img_path)
        axes[1, i].imshow(img)
        axes[1, i].set_title(f'Real Image {i+1}')
        axes[1, i].axis('off')
    
    plt.show()

# Example usage
fake_dir = '/kaggle/working/balanced_dataset/fake'
real_dir = '/kaggle/working/balanced_dataset/real'
display_first_5_images(fake_dir, real_dir)